# ── 0. 配置区 ──────────────────────────────────────────────

In [ ]:
from pathlib import Path

# FiftyOne 数据集和检测字段
DATASET_NAME = "test_run2_image_copy"
PRED_FIELD   = "pred_yolo11l_20pct_null_images_add_rawData_batch_4_final"

# 分类模型路径和设备
CLF_MODEL_PATH = Path("/home/tianqi/D/01_Projects/01_swd/02_code/pipeline/ultralytics_ty/_ty/02_fine-tuned_checkpoint/best_models/91_swd_cls/yolo11m-cls-best_v7.pt")
CLF_DEVICE     = "cuda"

# 分类过滤参数
TARGET_LABEL = "swd"
CLF_THRESH   = 0.5
PAD_RATIO    = 0.2
BATCH_SIZE   = 64

# ── 1. 初始化日志 ──────────────────────────────────────────

In [ ]:
import logging
import ipynbname

log_file_name = ipynbname.name() + ".log"
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    handlers=[
        logging.FileHandler(log_file_name, encoding="utf-8"),
        logging.StreamHandler(),
    ],
)
logger = logging.getLogger(__name__)
logger.info("============ Notebook logging 初始化完成 ============")

# ── 2. Step 1：加载 FiftyOne 数据集 ─────────────────────────

In [ ]:
# ── Step 1：加载数据集，打印基本信息 ────────────────────────
# 输入：DATASET_NAME
# 输出：dataset 对象

logger.info("Step 1 开始：加载 FiftyOne 数据集")

import fiftyone as fo
from IPython.display import display

try:
    dataset = fo.load_dataset(DATASET_NAME)
    logger.info(f"数据集加载成功：{DATASET_NAME}，共 {len(dataset)} 张")
    print(dataset)
except Exception as e:
    logger.error(f"数据集加载失败 {DATASET_NAME}: {e}")
    raise

logger.info("Step 1 完成：数据集加载成功")

# ── 3. Step 2：应用分类模型二次过滤检测框 ────────────────────

In [ ]:
# ── Step 2 辅助：构建分类预测函数 ────────────────────────────
from ultralytics import YOLO

clf_model = YOLO(str(CLF_MODEL_PATH))
logger.info(f"分类模型加载完成：{CLF_MODEL_PATH.name}，类别={clf_model.names}")


def clf_predict_fn(crops):
    """输入 list[PIL.Image]，返回 list[(label, score)]。"""
    results = clf_model.predict(source=crops, device=CLF_DEVICE, verbose=False)
    out = []
    for r in results:
        top1_idx = int(r.probs.top1)
        score    = float(r.probs.top1conf)
        label    = r.names[top1_idx]
        out.append((label, score))
    return out

In [ ]:
# ── Step 2：对每个 sample 应用分类过滤，结果写入 out_field ────
# 输入：dataset[PRED_FIELD] 中的检测框
# 输出：dataset[out_field] 仅保留分类为 TARGET_LABEL 且分数 >= CLF_THRESH 的框

logger.info("Step 2 开始：分类模型二次过滤")

from ty_nogt_tools import classify_and_filter_sample

out_field = PRED_FIELD + "_clf"

if out_field not in dataset.get_field_schema():
    dataset.add_sample_field(out_field, fo.EmbeddedDocumentField, embedded_doc_type=fo.Detections)

for sample in dataset.iter_samples(progress=True, autosave=True):
    try:
        classify_and_filter_sample(
            sample,
            pred_field=PRED_FIELD,
            out_field=out_field,
            clf_predict_fn=clf_predict_fn,
            target_label=TARGET_LABEL,
            clf_thresh=CLF_THRESH,
            pad_ratio=PAD_RATIO,
            batch_size=BATCH_SIZE,
        )
    except Exception as e:
        logger.error(f"分类过滤失败 {sample.filepath}: {e}")

logger.info(f"Step 2 完成：结果写入字段 {out_field}")

# ── 4. 验证 ────────────────────────────────────────────────

In [ ]:
# ── 验证：对比过滤前后检测数量 ───────────────────────────────
before_count = dataset.count(f"{PRED_FIELD}.detections")
after_count  = dataset.count(f"{out_field}.detections")

logger.info(f"验证：过滤前={before_count}，过滤后={after_count}，保留率={after_count/max(before_count,1)*100:.1f}%")
print(f"过滤前检测框总数: {before_count}")
print(f"过滤后检测框总数: {after_count}")
print(f"保留率: {after_count / max(before_count, 1) * 100:.1f}%")

# ── [可选] 按像素面积预过滤 ─────────────────────────────────

In [ ]:
# 可选：先按绝对像素面积过滤（95^2 ~ 500^2 px），再在此 view 上观察
from fiftyone import ViewField as F

dataset.compute_metadata()
bbox_area_px = (
    F("$metadata.width")  * F("bounding_box")[2] *
    F("$metadata.height") * F("bounding_box")[3]
)
filtered_view = dataset.filter_labels(
    PRED_FIELD,
    (95 ** 2 < bbox_area_px) & (bbox_area_px < 500 ** 2)
)
print(filtered_view)

# ── [可选] 分类模型独立测试（对图片文件夹批量推理）────────────

In [ ]:
# 可选：对指定文件夹下所有图片批量分类，输出结果 DataFrame
import os
import pandas as pd
from ultralytics import YOLO

STANDALONE_IMAGE_FOLDER = Path("/home/tianqi/D/01_Projects/01_swd/02_code/pipeline/ultralytics_ty/_ty/00_pipeline/00_temp/patch_data/swd")
STANDALONE_CONF_THR     = 0.30
IMG_EXT = {".jpg", ".jpeg", ".png"}

standalone_model = YOLO(str(CLF_MODEL_PATH))
image_paths = [p for p in STANDALONE_IMAGE_FOLDER.rglob("*") if p.suffix.lower() in IMG_EXT]
print(f"找到 {len(image_paths)} 张图片")

rows = []
for img_path in image_paths:
    rs = standalone_model(str(img_path), verbose=False)
    probs   = rs[0].probs.data.cpu().numpy()
    top1_i  = int(probs.argmax())
    top1_c  = float(probs[top1_i])
    label   = standalone_model.names[top1_i]
    rows.append({"filename": img_path.name, "pred": label, "conf": f"{top1_c:.4f}",
                 "status": "OK" if top1_c >= STANDALONE_CONF_THR else "low_conf"})

df_standalone = pd.DataFrame(rows)
display(df_standalone)